<a href="https://colab.research.google.com/github/Tazin17/pain-intensity-prediction-from-laser-evoked-eeg-responses/blob/main/exp8_classifier_swap_within_loocv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exp8 within-individual classifier-swap experiments

This notebook runs two controlled within-individual LOOCV experiments using only files already saved in Google Drive:

1. **V7 handcrafted 262 features + Huang Gaussian Naive Bayes**
2. **Huang CSP + Equation-3 beta1–beta5 features + V7 Extra Trees**

Both analyses use the same strict eligibility rule: each subject must contain at least **2 low-pain and 2 high-pain trials**. No raw `.set` files are re-extracted.

> Important: for Experiment 2, Huang beta features are recomputed inside every LOOCV fold from the saved `X_signal.npy`. The previously saved held-out beta CSV cannot safely be reused as a single training matrix because every held-out row was generated using a different fold-specific CSP and N2/P2 template.


In [ ]:

# CELL 1 — Mount Drive and imports

from google.colab import drive
drive.mount("/content/drive")

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.linalg import eigh

from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
)

print("Imports completed.")


In [ ]:

# CELL 2 — Shared metrics, eligibility, checkpoint, and summary helpers

def calculate_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    both_classes = len(np.unique(y_true)) == 2

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": (
            balanced_accuracy_score(y_true, y_pred)
            if both_classes else np.nan
        ),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1_score": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "roc_auc": (
            roc_auc_score(y_true, y_prob)
            if both_classes else np.nan
        ),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "specificity": (
            tn / (tn + fp) if (tn + fp) > 0 else np.nan
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def get_strict_eligible_subjects(y, groups, expected=192):
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups).astype(str)

    balance = (
        pd.DataFrame({
            "subject_id": groups,
            "pain_label": y,
        })
        .groupby(["subject_id", "pain_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[0, 1], fill_value=0)
    )
    balance.columns = ["n_low", "n_high"]

    eligible = balance.index[
        (balance["n_low"] >= 2)
        & (balance["n_high"] >= 2)
    ].astype(str).to_numpy()

    eligible = np.sort(eligible)

    if expected is not None and len(eligible) != expected:
        raise ValueError(
            f"Expected {expected} strict-eligible subjects, "
            f"but found {len(eligible)}."
        )

    return eligible, balance.reset_index()


def append_rows(rows_df, path):
    rows_df.to_csv(
        path,
        mode="a",
        header=not os.path.exists(path),
        index=False,
    )


def prepare_output_files(paths, overwrite=False):
    if overwrite:
        for path in paths:
            if os.path.exists(path):
                os.remove(path)


def make_final_outputs(
    prediction_path,
    subject_path,
    summary_path,
    experiment_name,
    feature_set,
    classifier_name,
):
    prediction_df = pd.read_csv(prediction_path)
    subject_df = pd.read_csv(subject_path)

    pooled = calculate_metrics(
        prediction_df["y_true"],
        prediction_df["y_pred"],
        prediction_df["y_prob"],
    )

    metric_columns = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc",
        "mcc",
        "cohen_kappa",
        "specificity",
    ]

    summary = {
        "experiment": experiment_name,
        "feature_set": feature_set,
        "classifier": classifier_name,
        "validation": (
            "Within-individual leave-one-trial-out CV; "
            "strict >=2 trials per class"
        ),
        "subjects_used": int(
            subject_df["subject_id"].nunique()
        ),
        "trials_used": int(len(prediction_df)),
    }

    for metric in metric_columns:
        summary[f"pooled_{metric}"] = pooled[metric]
        summary[f"mean_subject_{metric}"] = (
            subject_df[metric].mean()
        )
        summary[f"sd_subject_{metric}"] = (
            subject_df[metric].std(ddof=1)
        )

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(summary_path, index=False)

    print("\nPooled results:")
    display(
        pd.DataFrame([pooled])[
            [
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall",
                "f1_score",
                "roc_auc",
                "mcc",
                "cohen_kappa",
                "specificity",
            ]
        ]
    )

    print("\nSubject-wise mean ± SD:")
    for metric in metric_columns:
        mean_value = subject_df[metric].mean()
        sd_value = subject_df[metric].std(ddof=1)
        print(
            f"{metric:20s}: "
            f"{mean_value * 100:6.2f}% ± "
            f"{sd_value * 100:6.2f}%"
        )

    print("\nSaved summary:", summary_path)

    return prediction_df, subject_df, summary_df


In [ ]:

# CELL 3 — Experiment 1 configuration:
# V7 handcrafted 262 features + Huang Gaussian Naive Bayes

V5_DIR = (
    "/content/drive/MyDrive/"
    "EXP8_CLASSIFICATION_FROM_SCRATCH/"
    "V5_MAIN_PIPELINE_12CH"
)

X_V7_PATH = os.path.join(
    V5_DIR, "X_v5_handcrafted_all_262.npy"
)
Y_V7_PATH = os.path.join(
    V5_DIR, "y_v5_binary.npy"
)
GROUPS_V7_PATH = os.path.join(
    V5_DIR, "groups_v5_subject.npy"
)

V7_GNB_DIR = (
    "/content/drive/MyDrive/"
    "EXP8_CLASSIFICATION_FROM_SCRATCH/"
    "CLASSIFIER_SWAP_WITHIN/"
    "V7_FEATURES_HUANG_GNB"
)
os.makedirs(V7_GNB_DIR, exist_ok=True)

V7_GNB_PRED_PATH = os.path.join(
    V7_GNB_DIR, "v7_features_huang_gnb_predictions.csv"
)
V7_GNB_SUBJECT_PATH = os.path.join(
    V7_GNB_DIR, "v7_features_huang_gnb_subject_results.csv"
)
V7_GNB_SUMMARY_PATH = os.path.join(
    V7_GNB_DIR, "v7_features_huang_gnb_summary.csv"
)
V7_GNB_ELIGIBILITY_PATH = os.path.join(
    V7_GNB_DIR, "v7_features_huang_gnb_eligibility.csv"
)

# Set True only when you intentionally want a complete rerun.
OVERWRITE_V7_GNB = False

for required_path in [
    X_V7_PATH,
    Y_V7_PATH,
    GROUPS_V7_PATH,
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(required_path)

X_v7 = np.load(X_V7_PATH, mmap_mode="r")
y_v7 = np.load(Y_V7_PATH).astype(int)
groups_v7 = np.load(
    GROUPS_V7_PATH, allow_pickle=True
).astype(str)

if not (
    X_v7.shape[0]
    == len(y_v7)
    == len(groups_v7)
):
    raise ValueError("V7 saved files are not row-aligned.")

if not np.isfinite(np.asarray(X_v7[:100])).all():
    raise ValueError(
        "Non-finite values detected in the initial V7 feature check."
    )

eligible_v7, eligibility_v7 = (
    get_strict_eligible_subjects(
        y_v7, groups_v7, expected=192
    )
)
eligibility_v7.to_csv(
    V7_GNB_ELIGIBILITY_PATH, index=False
)

print("V7 feature shape:", X_v7.shape)
print("V7 class counts:", np.bincount(y_v7))
print("Strict eligible subjects:", len(eligible_v7))
print("Output folder:", V7_GNB_DIR)


In [ ]:

# CELL 4 — Run Experiment 1:
# V7 features + GaussianNB, within-individual LOOCV

prepare_output_files(
    [
        V7_GNB_PRED_PATH,
        V7_GNB_SUBJECT_PATH,
        V7_GNB_SUMMARY_PATH,
    ],
    overwrite=OVERWRITE_V7_GNB,
)

if os.path.exists(V7_GNB_SUBJECT_PATH):
    completed_subjects = set(
        pd.read_csv(
            V7_GNB_SUBJECT_PATH,
            dtype={"subject_id": str},
        )["subject_id"].astype(str)
    )
else:
    completed_subjects = set()

print(
    "Already completed subjects:",
    len(completed_subjects),
)

for subject_id in tqdm(
    eligible_v7,
    desc="V7 features + Huang GNB",
):
    subject_id = str(subject_id)

    if subject_id in completed_subjects:
        continue

    subject_indices = np.flatnonzero(
        groups_v7 == subject_id
    )

    subject_prediction_rows = []

    for test_index in subject_indices:
        train_indices = subject_indices[
            subject_indices != test_index
        ]

        y_train = y_v7[train_indices]
        y_test = y_v7[[test_index]]

        if len(np.unique(y_train)) != 2:
            raise RuntimeError(
                f"One-class training fold: "
                f"subject={subject_id}, row={test_index}"
            )

        # Exact Huang classifier swap:
        # default GaussianNB, no scaler and no PCA.
        model = GaussianNB()
        model.fit(
            X_v7[train_indices],
            y_train,
        )

        y_pred = int(
            model.predict(
                X_v7[[test_index]]
            )[0]
        )

        positive_position = int(
            np.flatnonzero(
                model.classes_ == 1
            )[0]
        )
        y_prob = float(
            model.predict_proba(
                X_v7[[test_index]]
            )[0, positive_position]
        )

        subject_prediction_rows.append({
            "subject_id": subject_id,
            "original_row_index": int(test_index),
            "y_true": int(y_test[0]),
            "y_pred": y_pred,
            "y_prob": y_prob,
        })

    subject_prediction_df = pd.DataFrame(
        subject_prediction_rows
    )

    subject_metrics = calculate_metrics(
        subject_prediction_df["y_true"],
        subject_prediction_df["y_pred"],
        subject_prediction_df["y_prob"],
    )

    subject_result_df = pd.DataFrame([{
        "subject_id": subject_id,
        "n_trials": int(
            len(subject_prediction_df)
        ),
        "n_low": int(
            np.sum(
                subject_prediction_df["y_true"] == 0
            )
        ),
        "n_high": int(
            np.sum(
                subject_prediction_df["y_true"] == 1
            )
        ),
        **subject_metrics,
    }])

    # A subject is checkpointed only after every one of
    # that subject's held-out trials has completed.
    append_rows(
        subject_prediction_df,
        V7_GNB_PRED_PATH,
    )
    append_rows(
        subject_result_df,
        V7_GNB_SUBJECT_PATH,
    )

    completed_subjects.add(subject_id)

v7_gnb_predictions, v7_gnb_subjects, v7_gnb_summary = (
    make_final_outputs(
        prediction_path=V7_GNB_PRED_PATH,
        subject_path=V7_GNB_SUBJECT_PATH,
        summary_path=V7_GNB_SUMMARY_PATH,
        experiment_name="V7 features + Huang GNB",
        feature_set="V7 handcrafted 262 features",
        classifier_name="Gaussian Naive Bayes",
    )
)



## Experiment 2 warning

Do **not** load `EXP8_Huang_within_192_beta_features.csv` and perform LOOCV directly on that whole CSV. Each saved row represents a held-out trial whose beta coefficients were generated by a different training-only CSP and template. The code below correctly loads the saved preprocessed EEG arrays and reconstructs both training and test beta features inside each fold.


In [ ]:

# CELL 5 — Experiment 2 configuration and Huang feature functions:
# Huang CSP + beta1–beta5 + V7 Extra Trees

HUANG_SAVE_DIR = (
    "/content/drive/MyDrive/"
    "EXP8_HUANG_EQ3_CLASSIFICATION_CORRECTED"
)

X_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "X_signal.npy"
)
Y_RATING_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "y_rating.npy"
)
Y_LABEL_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "y_label.npy"
)
LASER_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "laser_power.npy"
)
GROUPS_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "groups.npy"
)
TRIAL_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "trial_num.npy"
)
TIMES_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "times.npy"
)
CHANNELS_HUANG_PATH = os.path.join(
    HUANG_SAVE_DIR, "channels.csv"
)

HUANG_ET_DIR = (
    "/content/drive/MyDrive/"
    "EXP8_CLASSIFICATION_FROM_SCRATCH/"
    "CLASSIFIER_SWAP_WITHIN/"
    "HUANG_FEATURES_V7_EXTRATREES"
)
os.makedirs(HUANG_ET_DIR, exist_ok=True)

HUANG_ET_PRED_PATH = os.path.join(
    HUANG_ET_DIR,
    "huang_features_v7_extratrees_predictions.csv",
)
HUANG_ET_SUBJECT_PATH = os.path.join(
    HUANG_ET_DIR,
    "huang_features_v7_extratrees_subject_results.csv",
)
HUANG_ET_SUMMARY_PATH = os.path.join(
    HUANG_ET_DIR,
    "huang_features_v7_extratrees_summary.csv",
)
HUANG_ET_BETA_PATH = os.path.join(
    HUANG_ET_DIR,
    "huang_features_v7_extratrees_heldout_betas.csv",
)
HUANG_ET_ELIGIBILITY_PATH = os.path.join(
    HUANG_ET_DIR,
    "huang_features_v7_extratrees_eligibility.csv",
)

# Set True only for a complete rerun.
OVERWRITE_HUANG_ET = False

required_huang_paths = [
    X_HUANG_PATH,
    Y_RATING_HUANG_PATH,
    Y_LABEL_HUANG_PATH,
    LASER_HUANG_PATH,
    GROUPS_HUANG_PATH,
    TRIAL_HUANG_PATH,
    TIMES_HUANG_PATH,
    CHANNELS_HUANG_PATH,
]

for required_path in required_huang_paths:
    if not os.path.exists(required_path):
        raise FileNotFoundError(required_path)

X_huang = np.load(
    X_HUANG_PATH, mmap_mode="r"
)
y_rating_huang = np.load(
    Y_RATING_HUANG_PATH
)
y_huang = np.load(
    Y_LABEL_HUANG_PATH
).astype(int)
laser_huang = np.load(
    LASER_HUANG_PATH
)
groups_huang = np.load(
    GROUPS_HUANG_PATH,
    allow_pickle=True,
).astype(str)
trial_huang = np.load(
    TRIAL_HUANG_PATH
)
times_huang = np.load(
    TIMES_HUANG_PATH
)
channels_huang = pd.read_csv(
    CHANNELS_HUANG_PATH
)["channel"].tolist()

if not (
    X_huang.shape[0]
    == len(y_rating_huang)
    == len(y_huang)
    == len(laser_huang)
    == len(groups_huang)
    == len(trial_huang)
):
    raise ValueError(
        "Huang saved files are not row-aligned."
    )

if "Cz" not in channels_huang:
    raise ValueError(
        "Cz is absent from channels.csv."
    )

cz_idx = channels_huang.index("Cz")

# Same temporal settings as the uploaded Huang notebook.
pre_window = (-0.5, 0.0)
post_window = (0.0, 0.5)
n2_window = (0.150, 0.300)
p2_window = (0.300, 0.450)
n_csp_components = 3

pre_mask = (
    (times_huang >= pre_window[0])
    & (times_huang < pre_window[1])
)
post_mask = (
    (times_huang >= post_window[0])
    & (times_huang <= post_window[1])
)
post_times = times_huang[post_mask]


def mean_cov(trials):
    covariances = []

    for epoch in trials:
        covariance = epoch @ epoch.T
        trace_value = np.trace(covariance)

        if trace_value > 0:
            covariance = (
                covariance / trace_value
            )

        covariances.append(covariance)

    return np.mean(
        covariances, axis=0
    )


def fit_huang_csp(X_train):
    X_pre = X_train[:, :, pre_mask]
    X_post = X_train[:, :, post_mask]

    C_pre = mean_cov(X_pre)
    C_post = mean_cov(X_post)

    eigenvalues, eigenvectors = eigh(
        C_post, C_pre
    )

    order = np.argsort(
        eigenvalues
    )[::-1]
    selected = order[
        :n_csp_components
    ]

    W_full = eigenvectors
    W_selected = eigenvectors[
        :, selected
    ]

    A_full = np.linalg.pinv(
        W_full.T
    )
    A_selected = A_full[
        :, selected
    ]

    return W_selected, A_selected


def apply_csp_reconstruct(
    X_data,
    W_selected,
    A_selected,
):
    projected = np.einsum(
        "cf,nft->nct",
        W_selected.T,
        X_data,
    )

    reconstructed = np.einsum(
        "fc,nct->nft",
        A_selected,
        projected,
    )

    return reconstructed


def make_np_templates(
    train_cz_post
):
    average_response = (
        train_cz_post.mean(axis=0)
    )

    y_n = np.zeros_like(
        average_response
    )
    y_p = np.zeros_like(
        average_response
    )

    n_mask = (
        (post_times >= n2_window[0])
        & (post_times <= n2_window[1])
    )
    p_mask = (
        (post_times >= p2_window[0])
        & (post_times <= p2_window[1])
    )

    y_n[n_mask] = average_response[n_mask]
    y_p[p_mask] = average_response[p_mask]

    y_n_derivative = np.gradient(y_n)
    y_p_derivative = np.gradient(y_p)

    design_matrix = np.column_stack([
        y_n,
        y_n_derivative,
        y_p,
        y_p_derivative,
        np.ones_like(y_n),
    ])

    for column_index in range(4):
        column = design_matrix[
            :, column_index
        ]
        design_matrix[:, column_index] = (
            column - column.mean()
        ) / (
            column.std() + 1e-8
        )

    return design_matrix


BETA_NAMES = [
    "beta1_N2_amp",
    "beta2_N2_latency",
    "beta3_P2_amp",
    "beta4_P2_latency",
    "beta5_constant",
]


def extract_eq3_betas(
    cz_post_trials,
    design_matrix,
):
    beta_rows = []

    for signal in cz_post_trials:
        coefficients = np.linalg.lstsq(
            design_matrix,
            signal,
            rcond=None,
        )[0]
        beta_rows.append(coefficients)

    return np.asarray(
        beta_rows,
        dtype=float,
    )


eligible_huang, eligibility_huang = (
    get_strict_eligible_subjects(
        y_huang,
        groups_huang,
        expected=192,
    )
)
eligibility_huang.to_csv(
    HUANG_ET_ELIGIBILITY_PATH,
    index=False,
)

print("Huang signal shape:", X_huang.shape)
print("Huang class counts:", np.bincount(y_huang))
print("Channels:", len(channels_huang))
print("Strict eligible subjects:", len(eligible_huang))
print("Output folder:", HUANG_ET_DIR)


In [ ]:

# CELL 6 — Run Experiment 2:
# Huang beta features + exact V7 Extra Trees, within-individual LOOCV

prepare_output_files(
    [
        HUANG_ET_PRED_PATH,
        HUANG_ET_SUBJECT_PATH,
        HUANG_ET_SUMMARY_PATH,
        HUANG_ET_BETA_PATH,
    ],
    overwrite=OVERWRITE_HUANG_ET,
)

if os.path.exists(HUANG_ET_SUBJECT_PATH):
    completed_subjects = set(
        pd.read_csv(
            HUANG_ET_SUBJECT_PATH,
            dtype={"subject_id": str},
        )["subject_id"].astype(str)
    )
else:
    completed_subjects = set()

print(
    "Already completed subjects:",
    len(completed_subjects),
)

for subject_id in tqdm(
    eligible_huang,
    desc="Huang beta features + V7 Extra Trees",
):
    subject_id = str(subject_id)

    if subject_id in completed_subjects:
        continue

    subject_indices = np.flatnonzero(
        groups_huang == subject_id
    )

    # Only one subject is copied into RAM at a time.
    X_subject = np.asarray(
        X_huang[subject_indices],
        dtype=float,
    )
    y_subject = y_huang[
        subject_indices
    ]
    rating_subject = y_rating_huang[
        subject_indices
    ]
    laser_subject = laser_huang[
        subject_indices
    ]
    trial_subject = trial_huang[
        subject_indices
    ]

    prediction_rows = []
    heldout_beta_rows = []

    for test_local in range(
        len(subject_indices)
    ):
        train_local = np.delete(
            np.arange(len(subject_indices)),
            test_local,
        )

        X_train = X_subject[
            train_local
        ]
        X_test = X_subject[
            [test_local]
        ]
        y_train = y_subject[
            train_local
        ]
        y_test = y_subject[
            [test_local]
        ]

        if len(np.unique(y_train)) != 2:
            raise RuntimeError(
                f"One-class training fold: "
                f"subject={subject_id}, "
                f"local_test={test_local}"
            )

        # All Huang transformations are fit using
        # the current training trials only.
        W_selected, A_selected = (
            fit_huang_csp(X_train)
        )

        X_train_reconstructed = (
            apply_csp_reconstruct(
                X_train,
                W_selected,
                A_selected,
            )
        )
        X_test_reconstructed = (
            apply_csp_reconstruct(
                X_test,
                W_selected,
                A_selected,
            )
        )

        train_cz_post = (
            X_train_reconstructed[
                :, cz_idx, :
            ][:, post_mask]
        )
        test_cz_post = (
            X_test_reconstructed[
                :, cz_idx, :
            ][:, post_mask]
        )

        design_matrix = (
            make_np_templates(
                train_cz_post
            )
        )

        X_train_beta = (
            extract_eq3_betas(
                train_cz_post,
                design_matrix,
            )
        )
        X_test_beta = (
            extract_eq3_betas(
                test_cz_post,
                design_matrix,
            )
        )

        # Exact V7 Extra Trees hyperparameters.
        model = ExtraTreesClassifier(
            n_estimators=150,
            max_depth=12,
            min_samples_leaf=3,
            min_samples_split=5,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            X_train_beta,
            y_train,
        )

        y_pred = int(
            model.predict(
                X_test_beta
            )[0]
        )

        positive_position = int(
            np.flatnonzero(
                model.classes_ == 1
            )[0]
        )
        y_prob = float(
            model.predict_proba(
                X_test_beta
            )[0, positive_position]
        )

        original_row_index = int(
            subject_indices[test_local]
        )

        prediction_rows.append({
            "subject_id": subject_id,
            "original_row_index": original_row_index,
            "trial_num": trial_subject[test_local],
            "painrating": rating_subject[test_local],
            "pain_label": int(y_test[0]),
            "laser_power": laser_subject[test_local],
            "y_true": int(y_test[0]),
            "y_pred": y_pred,
            "y_prob": y_prob,
        })

        beta_row = {
            "subject_id": subject_id,
            "original_row_index": original_row_index,
            "trial_num": trial_subject[test_local],
            "painrating": rating_subject[test_local],
            "pain_label": int(y_test[0]),
        }

        for beta_name, beta_value in zip(
            BETA_NAMES,
            X_test_beta[0],
        ):
            beta_row[beta_name] = float(
                beta_value
            )

        heldout_beta_rows.append(
            beta_row
        )

    subject_prediction_df = pd.DataFrame(
        prediction_rows
    )
    subject_beta_df = pd.DataFrame(
        heldout_beta_rows
    )

    subject_metrics = calculate_metrics(
        subject_prediction_df["y_true"],
        subject_prediction_df["y_pred"],
        subject_prediction_df["y_prob"],
    )

    subject_result_df = pd.DataFrame([{
        "subject_id": subject_id,
        "n_trials": int(
            len(subject_prediction_df)
        ),
        "n_low": int(
            np.sum(
                subject_prediction_df["y_true"] == 0
            )
        ),
        "n_high": int(
            np.sum(
                subject_prediction_df["y_true"] == 1
            )
        ),
        **subject_metrics,
    }])

    append_rows(
        subject_prediction_df,
        HUANG_ET_PRED_PATH,
    )
    append_rows(
        subject_beta_df,
        HUANG_ET_BETA_PATH,
    )
    append_rows(
        subject_result_df,
        HUANG_ET_SUBJECT_PATH,
    )

    completed_subjects.add(subject_id)

huang_et_predictions, huang_et_subjects, huang_et_summary = (
    make_final_outputs(
        prediction_path=HUANG_ET_PRED_PATH,
        subject_path=HUANG_ET_SUBJECT_PATH,
        summary_path=HUANG_ET_SUMMARY_PATH,
        experiment_name="Huang features + V7 Extra Trees",
        feature_set="Huang CSP + Equation-3 beta1-beta5",
        classifier_name="V7 Extra Trees",
    )
)


In [ ]:

# CELL 7 — Display the two new classifier-swap results together

comparison_columns = [
    "experiment",
    "feature_set",
    "classifier",
    "subjects_used",
    "trials_used",
    "pooled_accuracy",
    "pooled_balanced_accuracy",
    "pooled_precision",
    "pooled_recall",
    "pooled_f1_score",
    "pooled_roc_auc",
    "mean_subject_accuracy",
    "sd_subject_accuracy",
]

swap_comparison = pd.concat(
    [
        pd.read_csv(V7_GNB_SUMMARY_PATH),
        pd.read_csv(HUANG_ET_SUMMARY_PATH),
    ],
    ignore_index=True,
)[comparison_columns]

percent_columns = [
    column
    for column in swap_comparison.columns
    if column.startswith("pooled_")
    or column.startswith("mean_subject_")
    or column.startswith("sd_subject_")
]

swap_comparison_percent = (
    swap_comparison.copy()
)

swap_comparison_percent[
    percent_columns
] = (
    swap_comparison_percent[
        percent_columns
    ] * 100
)

display(swap_comparison_percent)

COMPARISON_PATH = os.path.join(
    os.path.dirname(V7_GNB_DIR),
    "classifier_swap_within_comparison.csv",
)
swap_comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

print("Saved comparison:", COMPARISON_PATH)
